In [1]:
import mlflow
from pathlib import Path

from src.build_spark import spark_session_context

from src.training import predict_rul
from src.tracking import fetch_run_id
from src.data_processing import load_cmapss_raw

In [2]:
project_root = Path("/home/aanchal/nasa_c_mapss")

mlflow.set_tracking_uri("file://" + str(project_root / "mlruns"))

In [3]:
print(Path.cwd())
print(mlflow.get_tracking_uri())

/home/aanchal/nasa_c_mapss/code
file:///home/aanchal/nasa_c_mapss/mlruns


In [4]:
subset_id = "FD004"
unit_id = "5"

In [5]:
with spark_session_context(app_name="cmapss-rul-estimation") as spark:

    test_data = load_cmapss_raw(spark, project_root / "Data/CMAPSSData/test_FD004.txt")

    raw_trajectory = (test_data.where(f"unit_id = {unit_id}").orderBy("cycle"))

    prediction = predict_rul(subset_id=subset_id, 
                             training_run_id=fetch_run_id(subset_id, experiment_type="training", model_type="RandomForestRegressor"), 
                             raw_trajectory=raw_trajectory)

print(f"Predicted RUL: {prediction:.2f}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/20 14:33:42 WARN Utils: Your hostname, DESKTOP-HB4HHHI, resolves to a loopback address: 127.0.1.1; using 172.17.161.7 instead (on interface eth0)
26/08/20 14:33:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 14:33:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/20 14:33:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Predicted RUL: 68.03
